# 02 · Fine-tuning QLoRA — đa model VLM

Giai đoạn 2: chia dataset 80/10/10 → fine-tune QLoRA 4-bit.

**Notebook này chạy được cả 3 model** trong bộ so sánh — chỉ đổi `MODEL_KEY` ở cell đầu:

| `MODEL_KEY` | Model | GPU |
|---|---|---|
| `qwen` | Qwen/Qwen2.5-VL-3B-Instruct (anchor) | T4 / L4 |
| `internvl` | OpenGVLab/InternVL3_5-2B-HF | T4 đủ |
| `llama_vision` | meta-llama/Llama-3.2-11B-Vision-Instruct | **L4 24GB** + gated |

Mọi siêu tham số (LoRA r/alpha/dropout, `max_length`, tiền xử lý ảnh, target_modules)
đều lấy từ `src/models/vlm_registry.py` để **3 model dùng chung một cấu hình** —
điều kiện cần để so sánh công bằng. Xem `VLM_COMPARISON_PLAN.md`.

**Checkpoint adapter lưu thẳng vào Google Drive** → reconnect là train tiếp được.

> Runtime → Change runtime type → **GPU (L4)** trước khi chạy.

## 1. Mount Drive + đường dẫn dataset, checkpoint, model local

In [ ]:
# ═══ CẤU HÌNH — CHỈ CẦN ĐỔI 3 BIẾN NÀY ═══════════════════════════════════
# Model muốn fine-tune. Xem src/models/vlm_registry.py:
#   'qwen'         → Qwen/Qwen2.5-VL-3B-Instruct        (anchor, T4/L4)
#   'internvl'     → OpenGVLab/InternVL3_5-2B-HF        (T4 đủ)
#   'llama_vision' → meta-llama/Llama-3.2-11B-Vision-Instruct  (CẦN L4 24GB + gated)
MODEL_KEY = 'qwen'

SIDE = 'Back'          # 'Front' hoặc 'Back'

# Số epoch — BIẾN KIỂM SOÁT của thí nghiệm.
# Đổi ở đây thì PHẢI đổi giống nhau cho CẢ BA model, nếu không bảng so sánh mất
# tính công bằng. Tuyệt đối không chọn epoch khác nhau cho từng model.
EPOCHS = 3
# ═════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DRIVE = Path('/content/drive/MyDrive/cccd_project/Data')
REPO_DRIVE    = PROJECT_DRIVE / 'label_CCCD'

# Registry chỉ dùng thư viện chuẩn nên import được TRƯỚC khi pip install.
sys.path.insert(0, str(REPO_DRIVE))
from src.models.vlm_registry import resolve, resolve_model_dir

spec       = resolve(MODEL_KEY)
MODEL_ID   = spec.model_id
MODELS_DIR = PROJECT_DRIVE / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Nơi đặt snapshot base model do resolve_model_dir quyết định, KHÔNG hardcode:
# Llama-3.2-11B-Vision nặng ~21GB, không vừa Google Drive free (15GB) → tải nửa
# chừng rồi chết, để lại thư mục thiếu shard. Hàm này ưu tiên Drive (bền qua các
# lần Colab ngắt), nhưng rơi về SSD /content/models khi Drive không đủ chỗ, và
# luôn dùng lại nơi nào ĐÃ có snapshot đủ file.
MODEL_LOCAL = resolve_model_dir(spec, MODELS_DIR)
SSD_NOTE = '' if str(MODEL_LOCAL).startswith('/content/drive') else \
    '   ⚠ SSD tạm — mất khi ngắt phiên, notebook sau phải tải lại'

DATASET_DIR = PROJECT_DRIVE / 'dataset' / SIDE
AUG_DIR     = PROJECT_DRIVE / 'aug' / SIDE
REVIEWED    = PROJECT_DRIVE / 'draft' / f'{SIDE}_draft.jsonl'

# Quy ước tên checkpoint: {model_key}-cccd-lora-{side} — thống nhất cho cả 3 model
# và cho notebook 03/04/05 tự suy ra được.
CKPT_DIR = PROJECT_DRIVE / 'checkpoints' / f'{MODEL_KEY}-cccd-lora-{SIDE.lower()}'

for d in (DATASET_DIR, AUG_DIR, CKPT_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'🚀 Model      : {MODEL_ID}  [{spec.key} / {spec.family}]')
print(f'   Mặt thẻ    : {SIDE} | epochs={EPOCHS}')
print(f'   max_length : {spec.max_length}   (lấy từ registry)')
print(f'   vision     : {spec.vision_prefixes}  ← sẽ bị đóng băng, LoRA KHÔNG gắn vào')
print(f'   gated      : {spec.gated}')
print(f'📂 Base local : {MODEL_LOCAL}{SSD_NOTE}')
print(f'💾 Checkpoint : {CKPT_DIR}')
if spec.gated:
    print('\n⚠ Model GATED: phải accept license trên HuggingFace và đặt HF_TOKEN '
          'vào Colab Secrets (biểu tượng 🔑 bên trái) trước khi chạy cell tải model.')
if MODEL_KEY == 'llama_vision':
    print('⚠ 11B — cần GPU L4 24GB. T4 15GB sẽ OOM.')

## 2. Cài thư viện QLoRA (transformers bản ổn định cho Qwen2.5-VL)

In [ ]:
# --- 2. SETUP MÔI TRƯỜNG ---
!rm -rf /content/cccd
!mkdir -p /content/cccd
!cp -r /content/drive/MyDrive/cccd_project/Data/label_CCCD/* /content/cccd/ 2>/dev/null

%cd /content/cccd

# Phiên bản transformers tối thiểu lấy TỪ REGISTRY theo model đang chọn:
#   qwen >=4.49.0 | internvl >=4.52.1 | llama_vision >=4.45.0
# Cài thiếu version thì train.py sẽ dừng ngay ở check_available() với thông báo rõ.
!pip -q install 'transformers>={spec.min_transformers}' qwen-vl-utils accelerate peft bitsandbytes Pillow tqdm huggingface_hub
!nvidia-smi

import transformers
from src.models.vlm_registry import check_available
ok, reason = check_available(spec)
print(f'\ntransformers {transformers.__version__} | {spec.key} khả dụng: {ok} ({reason})')

## 3. Tải base model từ HuggingFace về LOCAL
Model nào là do `MODEL_KEY` quyết định. Nơi tải về do `resolve_model_dir` quyết định: **Drive** nếu còn đủ chỗ, **SSD `/content/models`** nếu không (Llama-11B ~21GB > Drive free 15GB).
Llama-3.2-Vision là model **gated** → cần `HF_TOKEN` trong Colab Secrets.


In [ ]:
# --- TẢI BASE MODEL VỀ LOCAL (đủ trọng số mới thôi) ---
# [ĐÃ SỬA] Trước đây guard bằng `if not (MODEL_LOCAL/'config.json').exists()`.
# snapshot_download tải file nhỏ (config, index, tokenizer) TRƯỚC, shard nặng SAU →
# lần tải bị đứt (hết chỗ trên Drive / Colab disconnect) vẫn để lại config.json,
# guard tưởng "đã có sẵn" nên bỏ qua, và lỗi chỉ nổ ra ở tận lúc nạp model:
#     FileNotFoundError: .../model-00001-of-00005.safetensors
# ensure_snapshot() đối chiếu model.safetensors.index.json nên biết thiếu shard nào
# (và bắt được cả shard ghi dở), rồi tải tiếp — snapshot_download resume được.
from src.models.vlm_registry import ensure_snapshot, snapshot_complete

ok, reason = snapshot_complete(MODEL_LOCAL)
print(f'Snapshot hiện có: {"đủ file" if ok else reason}  →  {MODEL_LOCAL}')

# Model gated (Llama-3.2-Vision) phải đăng nhập TRƯỚC khi tải. Đã có sẵn thì không cần.
if spec.gated and not ok:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(userdata.get('HF_TOKEN'))
        print('✓ Đã đăng nhập HuggingFace')
    except Exception as exc:
        raise SystemExit(
            f'{spec.model_id} là model gated nhưng chưa đăng nhập được ({exc}).\n'
            f'1) Vào https://huggingface.co/{spec.model_id} accept license\n'
            f'2) Tạo token tại https://huggingface.co/settings/tokens\n'
            f'3) Colab → 🔑 Secrets → thêm HF_TOKEN, bật "Notebook access"'
        )

ensure_snapshot(spec, MODEL_LOCAL)   # raise nếu tải xong mà vẫn thiếu file
print('✓ Base model sẵn sàng:', MODEL_LOCAL)


## 4. Chia dataset 80/10/10 (augment CHỈ train, chống leakage)

In [ ]:
# --- 4. CHIA DATASET (TÁCH RIÊNG TRAIN/VAL/TEST CHO MẶT ĐANG CHỌN) ---
!python -m src.data_pipeline.prepare_dataset \
    --input '{REVIEWED}' \
    --out_dir '{DATASET_DIR}' \
    --aug_dir '{AUG_DIR}' \
    --image_root '{PROJECT_DRIVE}' \
    --n_aug 2 --ratios 0.8,0.1,0.1 --seed 42

2026-06-26 07:46:10,051 [INFO] Đã load 2002 record hợp lệ
2026-06-26 07:46:10,063 [INFO] [train] 1602 ảnh
2026-06-26 07:46:10,063 [INFO] [val] 200 ảnh
2026-06-26 07:46:10,063 [INFO] [test] 200 ảnh
2026-06-26 07:46:10,077 [INFO] Đã ghi 200 record → /content/drive/MyDrive/cccd_project/Data/dataset/Back/val.jsonl
2026-06-26 07:46:10,090 [INFO] Đã ghi 200 record → /content/drive/MyDrive/cccd_project/Data/dataset/Back/test.jsonl
2026-06-26 10:38:25,185 [INFO] [train] augment: 1602 gốc → 4806 sau augment
2026-06-26 10:38:25,322 [INFO] Đã ghi 4806 record → /content/drive/MyDrive/cccd_project/Data/dataset/Back/train.jsonl
2026-06-26 10:38:25,322 [INFO] ✅ Hoàn tất. Dataset tại: /content/drive/MyDrive/cccd_project/Data/dataset/Back/


## 5. Fine-tune QLoRA (4-bit NF4 + gradient checkpointing) — base LOCAL
batch_size=1 + grad_accum=8 → effective batch 8, vừa T4 15GB.

In [ ]:
import json
import shutil
import os
from pathlib import Path
from tqdm.auto import tqdm

# [ĐÃ SỬA] SIDE và các đường dẫn Drive lấy TỪ CELL CẤU HÌNH ở mục 1 — KHÔNG
# khai báo lại ở đây. Trước đây cell này có dòng `SIDE = 'Back'` riêng: đổi SIDE ở
# cell đầu mà quên đổi ở đây thì ảnh + dataset copy lên SSD là của mặt KIA, trong
# khi CKPT_DIR vẫn mang tên mặt đã chọn → adapter train nhầm dữ liệu, và KHÔNG có
# lỗi nào báo ra. Giờ chỉ còn một nguồn duy nhất.
assert 'SIDE' in globals(), 'Chạy cell CẤU HÌNH ở mục 1 trước (nơi đặt MODEL_KEY / SIDE).'

DRIVE_IMG_DIR     = PROJECT_DRIVE / SIDE
DRIVE_AUG_DIR     = AUG_DIR          # = PROJECT_DRIVE/'aug'/SIDE
DRIVE_DATASET_DIR = DATASET_DIR      # = PROJECT_DRIVE/'dataset'/SIDE
print(f'📄 Mặt thẻ: {SIDE} | ảnh: {DRIVE_IMG_DIR} | dataset: {DRIVE_DATASET_DIR}')

SSD_IMAGES  = Path('/content/local_images')
SSD_DATASET = Path('/content/local_dataset')

SSD_IMAGES.mkdir(parents=True, exist_ok=True)
SSD_DATASET.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# BƯỚC 1 & 2: COPY ẢNH GỐC VÀ AUGMENT VÀO SSD
# ---------------------------------------------------------
print(f"\n🔄 1. Đang copy ảnh GỐC từ Drive -> SSD...")
copied_base = 0
if DRIVE_IMG_DIR.exists():
    for img in tqdm(list(DRIVE_IMG_DIR.glob('*.*')), desc="Ảnh Gốc"):
        if img.is_file() and img.stat().st_size > 0:
            shutil.copy(img, SSD_IMAGES / img.name)
            copied_base += 1

print(f"\n🔄 2. Đang copy ảnh AUGMENT từ Drive -> SSD...")
copied_aug = 0
if DRIVE_AUG_DIR.exists():
    for img in tqdm(list(DRIVE_AUG_DIR.glob('*.*')), desc="Ảnh Augment"):
        if img.is_file() and img.stat().st_size > 0:
            shutil.copy(img, SSD_IMAGES / img.name)
            copied_aug += 1

print(f"✅ Đã copy xong {copied_base + copied_aug} ảnh lên SSD!")

# ---------------------------------------------------------
# BƯỚC 3: CẬP NHẬT CẢ 3 FILE (TRAIN, VAL, TEST)
# ---------------------------------------------------------
print("\n🔄 3. Đang cập nhật và Validate các file JSONL...")

# Danh sách các file cần xử lý
splits = ['train.jsonl', 'val.jsonl', 'test.jsonl']

for split_name in splits:
    drive_file = DRIVE_DATASET_DIR / split_name
    ssd_file = SSD_DATASET / split_name

    if drive_file.exists():
        records = []
        missing = 0

        with open(drive_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            rec = json.loads(line)
            img_name = Path(rec['image']).name

            # Kiểm tra xem ảnh có trên SSD không
            if (SSD_IMAGES / img_name).exists():
                rec['image'] = img_name
                records.append(rec)
            else:
                missing += 1

        # Ghi ra SSD
        with open(ssd_file, 'w', encoding='utf-8') as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')

        print(f"  👉 [OK] {split_name}: {len(records)} records hợp lệ." +
              (f" (Bỏ qua {missing} lỗi)" if missing > 0 else ""))
    else:
        print(f"  ➖ [Bỏ qua] Không tìm thấy file {split_name} trên Drive.")

print(f"\n✅ HOÀN TẤT! Dữ liệu SSD đã sẵn sàng 100% cho mọi công đoạn.")


🔄 1. Đang copy ảnh GỐC từ Drive -> SSD...


Ảnh Gốc:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
# --- 4b. PROBE NGÂN SÁCH TOKEN ẢNH (rẻ: chỉ tải processor, KHÔNG tải trọng số) ---
# Chạy trước khi train để chắc chắn phần JSON đáp án không bị truncate.
#
# Cần xem: headroom > 256 token. Nếu âm/nhỏ → sửa trong vlm_registry (hạ
# max_patches với InternVL, hoặc nâng max_length), KHÔNG sửa riêng cho một model
# rồi bỏ qua các model khác.
#
# Đây cũng là cách đối chiếu ngân sách token ảnh giữa 3 model — điều kiện cần để
# so sánh công bằng. Chạy cell này cho cả 3 và ghi lại image_tokens.
import glob
_probe_img = sorted(glob.glob('/content/local_images/*'))[0]
!python -m src.models.vlm_registry --model '{MODEL_KEY}' --image '{_probe_img}'

In [ ]:
# --- 5. HUẤN LUYỆN QLoRA (base local + ổ SSD) ---
# KHÔNG truyền --max_length / --lora_dropout: để train.py lấy từ vlm_registry,
# đảm bảo cả 3 model dùng chung một ngân sách. --lora_r/--lora_alpha truyền tay
# cũng chỉ ghi đè đúng giá trị mặc định của registry (16/32).
#
# ✅ MỐC KIỂM CHỨNG trong log — phải thấy:
#    "✓ Vision tower (...) đã đóng băng hoàn toàn"   ← assert_no_vision_lora qua được
#    "Trainable params: N / ... (dưới 2%)"           ← LoRA CHỈ trên language model
#    Riêng Qwen2.5-VL-3B: N phải đúng bằng 29,933,568.
#    Nếu ra 37,152,768 là vision-LoRA vẫn còn → code chưa được cập nhật.
!python scripts/train.py \
    --train_jsonl '/content/local_dataset/train.jsonl' \
    --val_jsonl   '/content/local_dataset/val.jsonl' \
    --image_root  '/content/local_images' \
    --output_dir  '{CKPT_DIR}' \
    --model_name  '{MODEL_LOCAL}' \
    --model_key   '{MODEL_KEY}' \
    --epochs {EPOCHS} --batch_size 1 --grad_accum 8 --lr 1e-4 \
    --compute_dtype bfloat16 --lora_r 16 --lora_alpha 32